# Controlling Model Output

In [1]:
# Importing Necessary Modules
import boto3

In [3]:
# Creating the bedrock client
anthropic_bedrock = boto3.client('bedrock-runtime', region_name='us-west-2')

# Model ID
model_id = 'global.anthropic.claude-sonnet-4-6'

# System Prompt
system_prompt = ''

# Temperature 
temperature = 1.0

In [11]:
# Defining the function for user message
messages = []

def add_user_message(messages, text):
    user_message = {
        'role' : 'user',
        'content' : [
            { 'text' : text }
        ]
    }

    messages.append(user_message)
    return user_message

# Defining the function for assistance message
def add_assistant_message(messages, text):
    assistant_message = {
        'role' : 'assistant',
        'content' : [
            { 'text' : text }
        ]
    }

    messages.append(assistant_message)

# Function to invoke the model and return the assistant response
def chat(user_message, system=None, temperature = 1.0):

    params = {
        "modelId" : model_id,
        "messages" : messages,
        "inferenceConfig" : {
            'maxTokens' : 512,
            'temperature' : temperature
        }
    }

    if system:
        params["system"] = [{'text' : system}]

    response = anthropic_bedrock.converse(**params)

    return response['output']['message']['content'][0]['text']

### Prefilling Assistant Message

## In this case it says'claude does not support assistant message prefill

In [12]:
messages = []

add_user_message(messages, 'Is chai or coffee what is better')
add_assistant_message(messages, 'Chai is better for indians')

response = chat(messages)

response

ValidationException: An error occurred (ValidationException) when calling the Converse operation: The model returned the following errors: This model does not support assistant message prefill. The conversation must end with a user message.

## STOP Sequence

In [21]:
def chat(user_message, system=None, temperature = 1.0, stop_sequences=[]):
    params = {
        "modelId" : model_id,
        "messages" : messages,
        "inferenceConfig" : {
            'maxTokens' : 512,
            'temperature' : temperature,
            'stopSequences' : stop_sequences
        }
    }

    if system:
        params["system"] = [{'text' : system}]

    response = anthropic_bedrock.converse(**params)

    return response['output']['message']['content'][0]['text']

In [22]:
messages = []
add_user_message(messages, 'Count from 1 to 10')
response = chat(messages)
response

'Here is counting from 1 to 10:\n\n1, 2, 3, 4, 5, 6, 7, 8, 9, 10'

In [23]:
messages = []
add_user_message(messages, 'Count from 1 to 10')
response = chat(messages, stop_sequences = ['5'])
response

'Here is counting from 1 to 10:\n\n1, 2, 3, 4, '